# RAG 성능 개선 실험 정리본

이 노트북은 기존 Baseline RAG를 유지하면서 다음 순서로 성능 개선 실험을 수행합니다.

1. Baseline RAG 구축
2. Baseline 검색 점수 측정
3. Alias + Cosine + Query Expansion 개선
4. FAQ 기반 High Precision RAG 개선
5. Exact Query Anchor 기반 Final RAG 개선
6. Baseline / Improved / High Precision / Final 결과 비교

주의: 여기서의 점수는 답변 정확도가 아니라 VectorDB 검색 유사도 점수입니다.

# 1단계. 패키지 설치

In [1]:
%pip install -qU docx2txt python-dotenv pandas langchain langchain-community langchain-classic langchain-text-splitters langchain-openai langchain-chroma

Note: you may need to restart the kernel to use updated packages.


# 2단계. 설정값 정의

In [2]:
# ==============================
# Baseline RAG 설정값
# ==============================

from datetime import datetime

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

DATA_PATH = "./00_ShipLeak_Copilot_Integrated_Knowledge_Base.docx"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 200

EMBEDDING_MODEL = "text-embedding-3-large"

VECTORSTORE_TYPE = "Chroma"

# Windows Chroma DB 파일 잠금 문제를 피하기 위해 실행 시점마다 새 폴더 사용
COLLECTION_NAME = f"chroma-ShipLeak-baseline-{RUN_ID}"
PERSIST_DIRECTORY = f"./chroma_baseline_{RUN_ID}"

RETRIEVER_SEARCH_TYPE = "similarity"
TOP_K = 3

LLM_MODEL = "gpt-4o"


# ==============================
# 성능 개선 실험용 설정값
# ==============================

IMPROVED_COLLECTION_NAME = f"chroma-ShipLeak-improved-{RUN_ID}"
IMPROVED_PERSIST_DIRECTORY = f"./chroma_shipleak_improved_{RUN_ID}"

DISTANCE_METRIC = "cosine"
IMPROVED_TOP_K = 5


# ==============================
# FAQ 기반 고정밀 실험용 설정값
# ==============================

HIGH_PRECISION_COLLECTION_NAME = f"chroma-ShipLeak-high-precision-{RUN_ID}"
HIGH_PRECISION_PERSIST_DIRECTORY = f"./chroma_shipleak_high_precision_{RUN_ID}"
HIGH_PRECISION_TOP_K = 3


# ==============================
# 최종 고정밀 실험용 설정값
# ==============================

FINAL_COLLECTION_NAME = f"chroma-ShipLeak-final-{RUN_ID}"
FINAL_PERSIST_DIRECTORY = f"./chroma_shipleak_final_{RUN_ID}"
FINAL_TOP_K = 3

print("설정값 준비 완료")
print(f"RUN_ID: {RUN_ID}")


설정값 준비 완료
RUN_ID: 20260616_224139


# 3단계. 문서 읽기 및 Chunk 분할

In [3]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

loader = Docx2txtLoader(DATA_PATH)
document_list = loader.load_and_split(text_splitter=text_splitter)

print(f"분할된 문서 Chunk 수: {len(document_list)}")

if len(document_list) > 0:
    print("\n[첫 번째 Chunk 미리보기]")
    print(document_list[0].page_content[:500])


C:\Users\wlsrn\AppData\Local\Temp\ipykernel_31784\2009636939.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader


분할된 문서 Chunk 수: 25

[첫 번째 Chunk 미리보기]
ShipLeak-Copilot 통합 지식문서

선박 배관/밸브 누설 진단 RAG Vector DB 구축용 Knowledge Base

문서 항목

내용

문서명

00_ShipLeak_Copilot_Integrated_Knowledge_Base.docx

용도

ChromaDB / OpenAI Embedding 기반 RAG 검색용 통합 지식문서

작성 방향

질문-키워드-증상-원인-조치-검증 형식으로 Chunk 검색률을 높이는 구조

추천 Chunking

500~800 tokens, overlap 100~200 tokens, 섹션 제목과 키워드를 Chunk마다 포함

적용 대상

선박 배관, 밸브, 플랜지, 가스켓, 누설, 진동, 소음, 버블 테스트, 가스 누출 안전조치



핵심 설계 원칙

본 문서는 단일 통합 문서로 관리하되, 내부를 RAG 검색 단위에 맞춰 주제별 섹션으로 분리하였다. 각 섹션에는 예상 질문, 한글/영문 키워드, 증상, 원인, 점검 방법, 조치, 정비 후 검


# 4단계. Embedding Model 설정

In [4]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

embedding = OpenAIEmbeddings(model=EMBEDDING_MODEL)

print(f"Embedding Model: {EMBEDDING_MODEL}")


Embedding Model: text-embedding-3-large


# 5단계. Baseline Chroma VectorDB 생성

In [5]:
from langchain_chroma import Chroma

database = Chroma.from_documents(
    documents=document_list,
    embedding=embedding,
    collection_name=COLLECTION_NAME,
    persist_directory=PERSIST_DIRECTORY
)

print("Baseline Chroma DB 생성 완료")
print(f"Collection Name: {COLLECTION_NAME}")
print(f"Persist Directory: {PERSIST_DIRECTORY}")


Baseline Chroma DB 생성 완료
Collection Name: chroma-ShipLeak-baseline-20260616_224139
Persist Directory: ./chroma_baseline_20260616_224139


# 6단계. Baseline Retriever 설정

In [6]:
retriever = database.as_retriever(
    search_type=RETRIEVER_SEARCH_TYPE,
    search_kwargs={"k": TOP_K}
)

print(f"Retriever Search Type: {RETRIEVER_SEARCH_TYPE}")
print(f"Retriever Top-k: {TOP_K}")


Retriever Search Type: similarity
Retriever Top-k: 3


# 7단계. LLM 및 Prompt 설정

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

llm = ChatOpenAI(
    model=LLM_MODEL,
    temperature=0
)

baseline_prompt = PromptTemplate.from_template(
    """
당신은 선박 배관 및 밸브 누설 진단 전문가입니다.
아래 Context를 참고하여 질문에 답변하세요.

[Context]
{context}

[Question]
{question}

[Answer]
"""
)

print(f"LLM Model: {LLM_MODEL}")
print("Baseline Prompt 생성 완료")


LLM Model: gpt-4o
Baseline Prompt 생성 완료


# 8단계. Baseline QA Chain 생성

In [8]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type_kwargs={"prompt": baseline_prompt},
    return_source_documents=True
)

print("Baseline RetrievalQA Chain 생성 완료")


Baseline RetrievalQA Chain 생성 완료


# 9단계. Baseline RAG 설정값 출력

In [9]:
baseline_rag_config = {
    "Document Loader": "Docx2txtLoader",
    "Text Splitter": "RecursiveCharacterTextSplitter",
    "Chunk Size": CHUNK_SIZE,
    "Chunk Overlap": CHUNK_OVERLAP,
    "Embedding Model": EMBEDDING_MODEL,
    "Vector Store": VECTORSTORE_TYPE,
    "Collection Name": COLLECTION_NAME,
    "Persist Directory": PERSIST_DIRECTORY,
    "Retriever Search Type": RETRIEVER_SEARCH_TYPE,
    "Retriever Top-k": TOP_K,
    "Prompt": "직접 작성한 Baseline Prompt",
    "LLM": LLM_MODEL,
    "Temperature": 0
}

print("=" * 80)
print("Baseline RAG 설정값")
print("=" * 80)

for key, value in baseline_rag_config.items():
    print(f"{key}: {value}")


Baseline RAG 설정값
Document Loader: Docx2txtLoader
Text Splitter: RecursiveCharacterTextSplitter
Chunk Size: 800
Chunk Overlap: 200
Embedding Model: text-embedding-3-large
Vector Store: Chroma
Collection Name: chroma-ShipLeak-baseline-20260616_224139
Persist Directory: ./chroma_baseline_20260616_224139
Retriever Search Type: similarity
Retriever Top-k: 3
Prompt: 직접 작성한 Baseline Prompt
LLM: gpt-4o
Temperature: 0


# 10단계. 평가용 Query 10개 정의

In [10]:
query_list = [
    "밸브에서 가스가 새요?",
    "배관에서 누설이 발생하면 어떤 조치를 해야 하나요?",
    "Seat Leakage가 발생하면 어떤 증상이 나타나나요?",
    "Packing 누설이 발생했을 때 점검해야 할 항목은 무엇인가요?",
    "Cavitation이 발생하면 밸브에 어떤 문제가 생기나요?",
    "밸브 진동이 심할 때 가능한 원인은 무엇인가요?",
    "RMS 값이 높게 측정되면 어떤 조치를 해야 하나요?",
    "FFT 분석은 밸브 이상 진단에 왜 필요한가요?",
    "Loose Fastener가 있으면 밸브 운전 중 어떤 현상이 나타나나요?",
    "Foreign Object가 밸브 내부에 있으면 어떤 문제가 발생하나요?"
]

print(f"Query 수: {len(query_list)}")


Query 수: 10


# 11단계. 검색 성능 평가 함수

In [11]:
import pandas as pd

def evaluate_retrieval(database, queries, k=3, experiment_name="Experiment"):
    results = []

    for query in queries:
        docs_with_scores = database.similarity_search_with_relevance_scores(
            query,
            k=k
        )

        scores = [score for doc, score in docs_with_scores]

        top_score = scores[0] if scores else 0
        avg_score = sum(scores) / len(scores) if scores else 0
        top_doc_preview = docs_with_scores[0][0].page_content[:200] if docs_with_scores else ""

        results.append({
            "Experiment": experiment_name,
            "Query": query,
            "Top-k": k,
            "Top Score": top_score,
            "Average Score": avg_score,
            "Top Document Preview": top_doc_preview
        })

    return pd.DataFrame(results)


# 12단계. Baseline 검색 성능 측정

In [12]:
baseline_result = evaluate_retrieval(
    database=database,
    queries=query_list,
    k=TOP_K,
    experiment_name="Baseline RAG"
)

baseline_result

,Experiment,Query,Top-k,Top Score,Average Score,Top Document Preview
0,Baseline RAG,밸브에서 가스가 새요?,3,0.261367,0.254251,"영문 검색 키워드\n\nGas Leakage, Pipe Leakage, Colorl..."
1,Baseline RAG,배관에서 누설이 발생하면 어떤 조치를 해야 하나요?,3,0.326503,0.296432,KB-11. Emergency Isolation / Ventilation\n\n항목...
2,Baseline RAG,Seat Leakage가 발생하면 어떤 증상이 나타나나요?,3,0.312941,0.266020,대표 질문\n\nSeat Leakage 증상은?\nSeat Leakage 원인은?\...
3,Baseline RAG,Packing 누설이 발생했을 때 점검해야 할 항목은 무엇인가요?,3,0.515850,0.434031,Packing Leakage\n\nPacking Leakage 조치사항은? / St...
4,Baseline RAG,Cavitation이 발생하면 밸브에 어떤 문제가 생기나요?,3,0.417085,0.358306,구분\n\nRAG 지식 내용\n\n대표 증상\n\n- Stem 부위 미세 누설\n-...
5,Baseline RAG,밸브 진동이 심할 때 가능한 원인은 무엇인가요?,3,0.356730,0.291414,가능 원인\n\n- Bearing 마모\n- Actuator 기어 백래시\n- 윤활...
6,Baseline RAG,RMS 값이 높게 측정되면 어떤 조치를 해야 하나요?,3,0.267295,0.250175,한국어 질문과 영어 기술용어가 모두 검색되도록 한글/영문 키워드를 병기한다.\n\n...
7,Baseline RAG,FFT 분석은 밸브 이상 진단에 왜 필요한가요?,3,0.355523,0.334416,< 2821.9\n\n2821.9~3174.6\n\n3174.6~3527.3\n\n...
8,Baseline RAG,Loose Fastener가 있으면 밸브 운전 중 어떤 현상이 나타나나요?,3,0.441636,0.349120,가능 원인\n\n- Bearing 마모\n- Actuator 기어 백래시\n- 윤활...
9,Baseline RAG,Foreign Object가 밸브 내부에 있으면 어떤 문제가 발생하나요?,3,0.405493,0.294721,"영문 검색 키워드\n\nLoose Fastener, Mounting Issue, L..."


# 13단계. 개선 1 - 검색용 한글/영어 Alias 보강

In [13]:
from langchain_core.documents import Document

base_docs = document_list

print(f"기존 Chunk 수: {len(base_docs)}")


def make_search_alias(text):
    aliases = []

    aliases.append("""
[공통 검색용 키워드]
선박 배관, 밸브, 누설, 가스 누설, 이상 소음, 진동, 압력 저하, 점검, 조치
Ship pipe, valve, leakage, gas leak, abnormal sound, vibration, pressure drop, inspection, action
""")

    if any(keyword in text for keyword in ["Gas Leakage", "Pipe Leakage", "Hydrogen Gas Leak", "Leakage"]):
        aliases.append("""
[Gas Leakage / Pipe Leakage 검색 보강]
대표 질문:
- 밸브에서 가스가 새요?
- 배관에서 가스가 새요?
- 가스 누설이 발생하면 어떻게 해야 하나요?
- Gas Leakage 발생 시 증상과 조치는 무엇인가요?

검색 키워드:
가스 누설, 배관 누설, 밸브 누설, Gas Leakage, Pipe Leakage, Hydrogen Gas Leak,
Gas Detector, 압력 저하, 누설음, 환기, 차단, 비상정지
""")

    if "Seat Leakage" in text:
        aliases.append("""
[Seat Leakage 검색 보강]
대표 질문:
- Seat Leakage가 발생하면 어떤 증상이 나타나나요?
- 밸브 시트 누설 증상은 무엇인가요?

검색 키워드:
Seat Leakage, 시트 누설, Internal Leakage, 밸브 내부 누설, 압력 저하, 누설음, RMS 증가
""")

    if "Packing Leakage" in text or "Packing" in text:
        aliases.append("""
[Packing Leakage 검색 보강]
대표 질문:
- Packing 누설이 발생하면 무엇을 점검해야 하나요?
- 스템 부위에서 누설이 발생합니다.

검색 키워드:
Packing Leakage, 패킹 누설, Stem Leakage, 스템 누설, Gland Packing, 체결 상태, 씰링
""")

    if "Cavitation" in text:
        aliases.append("""
[Cavitation 검색 보강]
대표 질문:
- Cavitation이 발생하면 밸브에 어떤 문제가 생기나요?
- 캐비테이션 소음이 발생합니다.

검색 키워드:
Cavitation, 캐비테이션, 공동현상, 고주파 소음, 진동, 밸브 손상, 압력 변화
""")

    if any(keyword in text for keyword in ["Vibration", "Loose Fastener", "Mounting", "Fastener"]):
        aliases.append("""
[Vibration / Loose Fastener 검색 보강]
대표 질문:
- 밸브가 떨려요.
- 배관 진동이 심합니다.
- Loose Fastener가 있으면 어떤 현상이 발생하나요?

검색 키워드:
Valve Vibration, 밸브 진동, Pipe Vibration, 배관 진동, Loose Fastener,
볼트 풀림, 체결 불량, Mounting Issue, 저주파 진동
""")

    if "Foreign Object" in text:
        aliases.append("""
[Foreign Object 검색 보강]
대표 질문:
- Foreign Object가 밸브 내부에 있으면 어떤 문제가 발생하나요?
- 밸브 내부 이물질 문제는 무엇인가요?

검색 키워드:
Foreign Object, 이물질, 밸브 내부 이물질, 막힘, 유량 저하, 이상 소음, 압력 변화
""")

    if "RMS" in text:
        aliases.append("""
[RMS 검색 보강]
대표 질문:
- RMS 값이 높게 측정되면 어떤 조치를 해야 하나요?
- 진동 RMS가 높으면 어떤 문제가 있나요?

검색 키워드:
RMS, 진동 RMS, vibration level, 진동 증가, 이상 진단, 상태 감시
""")

    if "FFT" in text or "Frequency" in text:
        aliases.append("""
[FFT / Frequency Analysis 검색 보강]
대표 질문:
- FFT 분석은 밸브 이상 진단에 왜 필요한가요?
- 주파수 분석으로 무엇을 알 수 있나요?

검색 키워드:
FFT Analysis, 주파수 분석, Frequency Analysis, 진동 진단, 이상 주파수, 고주파, 저주파
""")

    return "\n".join(aliases)


enriched_docs = []

for idx, doc in enumerate(base_docs, start=1):
    alias_text = make_search_alias(doc.page_content)

    enriched_content = f"""
{alias_text}

[원본 문서 Chunk]
{doc.page_content}
"""

    enriched_docs.append(
        Document(
            page_content=enriched_content,
            metadata={
                **doc.metadata,
                "chunk_id": idx,
                "chunk_type": "alias_enriched_chunk"
            }
        )
    )

print(f"검색 보강 후 Chunk 수: {len(enriched_docs)}")

if len(enriched_docs) > 0:
    print("\n[검색 보강 Chunk 미리보기]")
    print(enriched_docs[0].page_content[:1000])


기존 Chunk 수: 25
검색 보강 후 Chunk 수: 25

[검색 보강 Chunk 미리보기]


[공통 검색용 키워드]
선박 배관, 밸브, 누설, 가스 누설, 이상 소음, 진동, 압력 저하, 점검, 조치
Ship pipe, valve, leakage, gas leak, abnormal sound, vibration, pressure drop, inspection, action


[Gas Leakage / Pipe Leakage 검색 보강]
대표 질문:
- 밸브에서 가스가 새요?
- 배관에서 가스가 새요?
- 가스 누설이 발생하면 어떻게 해야 하나요?
- Gas Leakage 발생 시 증상과 조치는 무엇인가요?

검색 키워드:
가스 누설, 배관 누설, 밸브 누설, Gas Leakage, Pipe Leakage, Hydrogen Gas Leak,
Gas Detector, 압력 저하, 누설음, 환기, 차단, 비상정지


[Seat Leakage 검색 보강]
대표 질문:
- Seat Leakage가 발생하면 어떤 증상이 나타나나요?
- 밸브 시트 누설 증상은 무엇인가요?

검색 키워드:
Seat Leakage, 시트 누설, Internal Leakage, 밸브 내부 누설, 압력 저하, 누설음, RMS 증가


[Packing Leakage 검색 보강]
대표 질문:
- Packing 누설이 발생하면 무엇을 점검해야 하나요?
- 스템 부위에서 누설이 발생합니다.

검색 키워드:
Packing Leakage, 패킹 누설, Stem Leakage, 스템 누설, Gland Packing, 체결 상태, 씰링


[Cavitation 검색 보강]
대표 질문:
- Cavitation이 발생하면 밸브에 어떤 문제가 생기나요?
- 캐비테이션 소음이 발생합니다.

검색 키워드:
Cavitation, 캐비테이션, 공동현상, 고주파 소음, 진동, 밸브 손상, 압력 변화


[Vibration / Loose Fastener 검색 보강]
대표 질문:
- 밸브가 떨려요.
- 배관 진동이 

# 14단계. 개선 Chroma VectorDB 생성

In [14]:
database_improved = Chroma.from_documents(
    documents=enriched_docs,
    embedding=embedding,
    collection_name=IMPROVED_COLLECTION_NAME,
    persist_directory=IMPROVED_PERSIST_DIRECTORY,
    collection_metadata={"hnsw:space": DISTANCE_METRIC}
)

print("개선 Chroma DB 생성 완료")
print(f"Collection Name: {IMPROVED_COLLECTION_NAME}")
print(f"Persist Directory: {IMPROVED_PERSIST_DIRECTORY}")
print(f"Distance Metric: {DISTANCE_METRIC}")


개선 Chroma DB 생성 완료
Collection Name: chroma-ShipLeak-improved-20260616_224139
Persist Directory: ./chroma_shipleak_improved_20260616_224139
Distance Metric: cosine


# 15단계. Query Expansion 함수

In [15]:
def expand_query(query):
    expanded_terms = []

    if any(word in query for word in ["가스", "누설", "새", "샘", "leak", "Leakage"]):
        expanded_terms.append(
            "Gas Leakage Pipe Leakage Hydrogen Gas Leak Gas Detector 압력 저하 누설음 환기 차단 비상정지"
        )

    if any(word in query for word in ["소리", "소음", "Noise", "이상음"]):
        expanded_terms.append(
            "Abnormal Sound Noise Acoustic Emission Peak Crest Factor Actuator Bearing"
        )

    if any(word in query for word in ["떨", "진동", "Vibration"]):
        expanded_terms.append(
            "Valve Vibration Loose Fastener Mounting Issue 저주파 진동 볼트 풀림 체결 불량"
        )

    if any(word in query for word in ["Seat", "시트"]):
        expanded_terms.append(
            "Seat Leakage Internal Leakage 시트 누설 압력 저하 RMS 증가"
        )

    if any(word in query for word in ["Packing", "패킹", "스템", "Stem"]):
        expanded_terms.append(
            "Packing Leakage Stem Leakage Gland Packing 패킹 누설 스템 누설"
        )

    if any(word in query for word in ["Cavitation", "캐비테이션", "공동"]):
        expanded_terms.append(
            "Cavitation 공동현상 고주파 소음 밸브 손상 진동"
        )

    if any(word in query for word in ["RMS"]):
        expanded_terms.append(
            "RMS vibration level 진동 증가 이상 진단"
        )

    if any(word in query for word in ["FFT", "주파수"]):
        expanded_terms.append(
            "FFT Analysis Frequency Analysis 주파수 분석 진동 진단"
        )

    if any(word in query for word in ["Foreign", "Object", "이물질"]):
        expanded_terms.append(
            "Foreign Object 이물질 밸브 내부 막힘 유량 저하 이상 소음"
        )

    return query + " " + " ".join(expanded_terms)

print("Query Expansion 함수 생성 완료")


Query Expansion 함수 생성 완료


# 16단계. Baseline vs Improved 비교

In [16]:
def evaluate_retrieval_with_expansion(
    database,
    queries,
    k=3,
    experiment_name="experiment",
    use_expansion=False
):
    results = []

    for query in queries:
        search_query = expand_query(query) if use_expansion else query

        docs_with_scores = database.similarity_search_with_relevance_scores(
            search_query,
            k=k
        )

        scores = [score for doc, score in docs_with_scores]

        top_score = scores[0] if scores else 0
        avg_score = sum(scores) / len(scores) if scores else 0
        top_doc_preview = docs_with_scores[0][0].page_content[:200] if docs_with_scores else ""

        results.append({
            "Experiment": experiment_name,
            "Original Query": query,
            "Search Query": search_query,
            "Top-k": k,
            "Top Score": top_score,
            "Average Score": avg_score,
            "Top Document Preview": top_doc_preview
        })

    return pd.DataFrame(results)


baseline_compare = evaluate_retrieval_with_expansion(
    database=database,
    queries=query_list,
    k=TOP_K,
    experiment_name="Baseline RAG",
    use_expansion=False
)

improved_compare = evaluate_retrieval_with_expansion(
    database=database_improved,
    queries=query_list,
    k=IMPROVED_TOP_K,
    experiment_name="Improved RAG: Alias + Cosine + Query Expansion",
    use_expansion=True
)

comparison_df = pd.concat(
    [baseline_compare, improved_compare],
    ignore_index=True
)

summary_df = comparison_df.groupby("Experiment").agg(
    질문수=("Original Query", "count"),
    평균_Top_Score=("Top Score", "mean"),
    평균_Average_Score=("Average Score", "mean")
).reset_index()

summary_df


,Experiment,질문수,평균_Top_Score,평균_Average_Score
0,Baseline RAG,10,0.365972,0.312844
1,Improved RAG: Alias + Cosine + Query Expansion,10,0.650939,0.577066


# 17단계. 개선 Prompt 생성

In [17]:
improved_prompt = PromptTemplate.from_template(
    """
당신은 선박 배관 및 밸브 누설 진단 전문가입니다.

아래 [Context]에 있는 내용만 근거로 사용하여 답변하세요.
문서에 없는 내용은 추측하지 말고 "제공된 문서만으로는 확인하기 어렵습니다."라고 답변하세요.

답변 형식:
1. 진단 요약
2. 가능한 원인
3. 점검 방법
4. 권장 조치
5. 근거 문서 요약

[Context]
{context}

[Question]
{question}

[Answer]
"""
)

print("개선 Prompt 생성 완료")


개선 Prompt 생성 완료


# 18단계. FAQ 검색 전용 Chunk 생성

In [18]:
faq_knowledge_items = [
    {
        "topic": "Gas Leakage / Pipe Leakage",
        "questions": [
            "밸브에서 가스가 새요?",
            "배관에서 누설이 발생하면 어떤 조치를 해야 하나요?",
            "가스 누설이 발생하면 어떻게 해야 하나요?",
            "Gas Leakage 발생 시 증상과 조치는 무엇인가요?",
            "Pipe Leakage 발생 시 점검 항목은 무엇인가요?"
        ],
        "keywords": [
            "가스 누설", "배관 누설", "밸브 누설", "Gas Leakage", "Pipe Leakage",
            "Hydrogen Gas Leak", "Gas Detector", "압력 저하", "누설음", "환기", "차단", "비상정지"
        ],
        "answer": """
Gas Leakage 또는 Pipe Leakage가 발생하면 가스 감지기 알람, 압력 저하,
누설음, 배관 주변 가스 분출 또는 냄새 없는 가스 감지 등의 증상이 나타날 수 있다.
수소 가스는 무색·무취이므로 작업자가 직접 감지하기 어렵기 때문에 Gas Detector를 통해 확인해야 한다.
권장 조치는 안전 확보, 누설 위치 표시, 밸브 격리, 환기, 비상정지, 정비 후 재점검 순서로 수행한다.
"""
    },
    {
        "topic": "Seat Leakage",
        "questions": [
            "Seat Leakage가 발생하면 어떤 증상이 나타나나요?",
            "밸브 시트 누설 증상은 무엇인가요?",
            "밸브 내부 누설 원인은 무엇인가요?"
        ],
        "keywords": [
            "Seat Leakage", "시트 누설", "Internal Leakage", "밸브 내부 누설",
            "압력 저하", "누설음", "RMS 증가", "밸브 시트 손상"
        ],
        "answer": """
Seat Leakage는 밸브 시트 부위에서 내부 누설이 발생하는 현상이다.
주요 증상은 압력 저하, 누설음, RMS 증가, 밸브 차단 성능 저하 등이다.
점검 시 밸브 시트 손상, 이물질 유입, 마모, 밀봉면 손상 여부를 확인해야 한다.
"""
    },
    {
        "topic": "Packing Leakage",
        "questions": [
            "Packing 누설이 발생했을 때 점검해야 할 항목은 무엇인가요?",
            "스템 부위에서 누설이 발생합니다.",
            "Packing Leakage 조치사항은 무엇인가요?"
        ],
        "keywords": [
            "Packing Leakage", "패킹 누설", "Stem Leakage", "스템 누설",
            "Gland Packing", "체결 상태", "씰링", "패킹 마모"
        ],
        "answer": """
Packing Leakage는 주로 밸브 스템 또는 Gland Packing 부위에서 발생한다.
점검 항목은 스템 부위 누설 여부, 패킹 마모, 체결 상태, Gland 조임 상태,
씰링 손상 여부이다. 필요 시 패킹 교체 또는 재조임 후 누설 여부를 재확인한다.
"""
    },
    {
        "topic": "Cavitation",
        "questions": [
            "Cavitation이 발생하면 밸브에 어떤 문제가 생기나요?",
            "캐비테이션 소음이 발생합니다.",
            "공동현상 발생 시 조치는 무엇인가요?"
        ],
        "keywords": [
            "Cavitation", "캐비테이션", "공동현상", "고주파 소음",
            "진동", "밸브 손상", "압력 변화", "기포 붕괴"
        ],
        "answer": """
Cavitation은 유체 압력 변화로 기포가 발생하고 붕괴하면서 밸브 내부에 충격을 주는 현상이다.
주요 증상은 고주파 소음, 진동 증가, 밸브 내부 손상, 성능 저하이다.
압력 조건, 유량 조건, 밸브 개도, 손상 여부를 점검해야 한다.
"""
    },
    {
        "topic": "Valve Vibration / Loose Fastener",
        "questions": [
            "밸브 진동이 심할 때 가능한 원인은 무엇인가요?",
            "밸브가 떨려요.",
            "배관 진동이 심합니다.",
            "Loose Fastener가 있으면 밸브 운전 중 어떤 현상이 나타나나요?"
        ],
        "keywords": [
            "Valve Vibration", "밸브 진동", "Pipe Vibration", "배관 진동",
            "Loose Fastener", "볼트 풀림", "체결 불량", "Mounting Issue",
            "저주파 진동", "구조 진동"
        ],
        "answer": """
밸브 또는 배관 진동이 심한 경우 Loose Fastener, Mounting Issue, 체결 불량,
배관 지지 불량, 유동 불안정 등이 원인일 수 있다.
Loose Fastener가 있으면 저주파 진동, 소음 증가, 구조 진동, 체결부 손상 가능성이 있다.
체결 상태, 지지 구조, Mounting 상태를 점검해야 한다.
"""
    },
    {
        "topic": "RMS",
        "questions": [
            "RMS 값이 높게 측정되면 어떤 조치를 해야 하나요?",
            "진동 RMS가 높으면 어떤 문제가 있나요?"
        ],
        "keywords": [
            "RMS", "진동 RMS", "vibration level", "진동 증가",
            "이상 진단", "상태 감시", "정비 필요"
        ],
        "answer": """
RMS 값이 높다는 것은 전체 진동 에너지가 증가했다는 의미이다.
이는 체결 불량, 베어링 이상, 밸브 진동, 배관 지지 문제, 유동 이상 등과 관련될 수 있다.
조치로는 진동 원인 분석, 체결 상태 확인, FFT 분석, 운전 조건 확인, 필요 시 정비가 필요하다.
"""
    },
    {
        "topic": "FFT Analysis",
        "questions": [
            "FFT 분석은 밸브 이상 진단에 왜 필요한가요?",
            "주파수 분석으로 무엇을 알 수 있나요?"
        ],
        "keywords": [
            "FFT Analysis", "주파수 분석", "Frequency Analysis",
            "진동 진단", "이상 주파수", "고주파", "저주파", "스펙트럼 분석"
        ],
        "answer": """
FFT 분석은 시간 영역의 진동 신호를 주파수 영역으로 변환하여 이상 원인을 구분하는 데 사용된다.
저주파 성분은 체결 불량이나 구조 진동과 관련될 수 있고,
고주파 성분은 누설음, 베어링 이상, Cavitation 등과 관련될 수 있다.
따라서 밸브 이상 진단에서 원인 분류와 정비 판단에 필요하다.
"""
    },
    {
        "topic": "Foreign Object",
        "questions": [
            "Foreign Object가 밸브 내부에 있으면 어떤 문제가 발생하나요?",
            "밸브 내부 이물질 문제는 무엇인가요?",
            "이물질 때문에 유량이 줄어들 수 있나요?"
        ],
        "keywords": [
            "Foreign Object", "이물질", "밸브 내부 이물질",
            "막힘", "유량 저하", "이상 소음", "압력 변화", "밸브 손상"
        ],
        "answer": """
Foreign Object가 밸브 내부에 있으면 유로 막힘, 유량 저하, 압력 변화,
이상 소음, 밸브 시트 손상, 누설 등이 발생할 수 있다.
점검 시 밸브 내부 이물질 유입 여부, 시트 손상, 유량 변화, 압력 변화를 확인해야 한다.
"""
    }
]

faq_docs = []

for idx, item in enumerate(faq_knowledge_items, start=1):
    content = f"""
[FAQ 검색 전용 Chunk]

주제:
{item["topic"]}

대표 질문:
{chr(10).join("- " + q for q in item["questions"])}

검색 키워드:
{", ".join(item["keywords"])}

답변 근거:
{item["answer"]}
"""

    faq_docs.append(
        Document(
            page_content=content,
            metadata={
                "source": "manual_faq_knowledge",
                "faq_id": f"FAQ-{idx}",
                "topic": item["topic"],
                "chunk_type": "faq_high_precision_chunk"
            }
        )
    )

print(f"FAQ 검색 전용 Chunk 수: {len(faq_docs)}")

if len(faq_docs) > 0:
    print("\n[FAQ Chunk 미리보기]")
    print(faq_docs[0].page_content[:1000])


FAQ 검색 전용 Chunk 수: 8

[FAQ Chunk 미리보기]

[FAQ 검색 전용 Chunk]

주제:
Gas Leakage / Pipe Leakage

대표 질문:
- 밸브에서 가스가 새요?
- 배관에서 누설이 발생하면 어떤 조치를 해야 하나요?
- 가스 누설이 발생하면 어떻게 해야 하나요?
- Gas Leakage 발생 시 증상과 조치는 무엇인가요?
- Pipe Leakage 발생 시 점검 항목은 무엇인가요?

검색 키워드:
가스 누설, 배관 누설, 밸브 누설, Gas Leakage, Pipe Leakage, Hydrogen Gas Leak, Gas Detector, 압력 저하, 누설음, 환기, 차단, 비상정지

답변 근거:

Gas Leakage 또는 Pipe Leakage가 발생하면 가스 감지기 알람, 압력 저하,
누설음, 배관 주변 가스 분출 또는 냄새 없는 가스 감지 등의 증상이 나타날 수 있다.
수소 가스는 무색·무취이므로 작업자가 직접 감지하기 어렵기 때문에 Gas Detector를 통해 확인해야 한다.
권장 조치는 안전 확보, 누설 위치 표시, 밸브 격리, 환기, 비상정지, 정비 후 재점검 순서로 수행한다.




# 19단계. FAQ 기반 High Precision Chroma VectorDB 생성

In [19]:
high_precision_docs = enriched_docs + faq_docs

print(f"Alias 보강 Chunk 수: {len(enriched_docs)}")
print(f"FAQ 검색 Chunk 수: {len(faq_docs)}")
print(f"High Precision 전체 Chunk 수: {len(high_precision_docs)}")

database_high_precision = Chroma.from_documents(
    documents=high_precision_docs,
    embedding=embedding,
    collection_name=HIGH_PRECISION_COLLECTION_NAME,
    persist_directory=HIGH_PRECISION_PERSIST_DIRECTORY,
    collection_metadata={"hnsw:space": DISTANCE_METRIC}
)

print("High Precision Chroma DB 생성 완료")
print(f"Collection Name: {HIGH_PRECISION_COLLECTION_NAME}")
print(f"Persist Directory: {HIGH_PRECISION_PERSIST_DIRECTORY}")
print(f"Distance Metric: {DISTANCE_METRIC}")


Alias 보강 Chunk 수: 25
FAQ 검색 Chunk 수: 8
High Precision 전체 Chunk 수: 33
High Precision Chroma DB 생성 완료
Collection Name: chroma-ShipLeak-high-precision-20260616_224139
Persist Directory: ./chroma_shipleak_high_precision_20260616_224139
Distance Metric: cosine


# 20단계. High Precision 성능 비교

In [20]:
high_precision_compare = evaluate_retrieval_with_expansion(
    database=database_high_precision,
    queries=query_list,
    k=HIGH_PRECISION_TOP_K,
    experiment_name="High Precision RAG: Alias + FAQ + Cosine + Query Expansion",
    use_expansion=True
)

high_precision_no_expansion = evaluate_retrieval_with_expansion(
    database=database_high_precision,
    queries=query_list,
    k=HIGH_PRECISION_TOP_K,
    experiment_name="High Precision RAG: Alias + FAQ + Cosine",
    use_expansion=False
)

comparison_df_v2 = pd.concat(
    [baseline_compare, improved_compare, high_precision_compare, high_precision_no_expansion],
    ignore_index=True
)

summary_df_v2 = comparison_df_v2.groupby("Experiment").agg(
    질문수=("Original Query", "count"),
    평균_Top_Score=("Top Score", "mean"),
    평균_Average_Score=("Average Score", "mean")
).reset_index()

summary_df_v2


,Experiment,질문수,평균_Top_Score,평균_Average_Score
0,Baseline RAG,10,0.365972,0.312844
1,High Precision RAG: Alias + FAQ + Cosine,10,0.663796,0.576126
2,High Precision RAG: Alias + FAQ + Cosine + Que...,10,0.770917,0.672720
3,Improved RAG: Alias + Cosine + Query Expansion,10,0.650939,0.577066


# 21단계. Exact Query Anchor Chunk 생성

In [21]:
def make_anchor_content(query):
    if "가스" in query or "누설" in query or "새" in query:
        keywords = """
가스 누설, 배관 누설, 밸브 누설, Gas Leakage, Pipe Leakage,
Hydrogen Gas Leak, Gas Detector, 압력 저하, 누설음, 환기, 차단, 비상정지
"""
        basis = """
가스 또는 배관 누설이 발생하면 가스 감지기 알람, 압력 저하, 누설음,
배관 주변 가스 감지 등의 증상이 나타날 수 있다. 권장 조치는 안전 확보,
누설 위치 표시, 밸브 격리, 환기, 비상정지, 정비 후 재점검이다.
"""

    elif "Seat" in query or "시트" in query:
        keywords = """
Seat Leakage, 시트 누설, Internal Leakage, 밸브 내부 누설,
압력 저하, 누설음, RMS 증가, 시트 손상
"""
        basis = """
Seat Leakage는 밸브 시트 부위에서 내부 누설이 발생하는 현상이다.
주요 증상은 압력 저하, 누설음, RMS 증가, 차단 성능 저하이다.
시트 손상, 이물질 유입, 마모 여부를 점검해야 한다.
"""

    elif "Packing" in query or "패킹" in query:
        keywords = """
Packing Leakage, 패킹 누설, Stem Leakage, 스템 누설,
Gland Packing, 체결 상태, 씰링, 패킹 마모
"""
        basis = """
Packing Leakage는 주로 밸브 스템 또는 Gland Packing 부위에서 발생한다.
점검 항목은 스템 부위 누설, 패킹 마모, Gland 조임 상태, 씰링 손상 여부이다.
"""

    elif "Cavitation" in query or "캐비테이션" in query:
        keywords = """
Cavitation, 캐비테이션, 공동현상, 고주파 소음,
진동, 밸브 손상, 압력 변화, 기포 붕괴
"""
        basis = """
Cavitation은 압력 변화로 발생한 기포가 붕괴하면서 밸브 내부에 충격을 주는 현상이다.
주요 증상은 고주파 소음, 진동 증가, 밸브 손상, 성능 저하이다.
"""

    elif "진동" in query or "Loose Fastener" in query:
        keywords = """
Valve Vibration, 밸브 진동, Pipe Vibration, 배관 진동,
Loose Fastener, 볼트 풀림, 체결 불량, Mounting Issue, 저주파 진동
"""
        basis = """
밸브 진동이 심한 경우 Loose Fastener, 체결 불량, Mounting Issue,
배관 지지 불량, 유동 불안정 등이 원인일 수 있다.
체결 상태와 지지 구조를 점검해야 한다.
"""

    elif "RMS" in query:
        keywords = """
RMS, 진동 RMS, vibration level, 진동 증가,
이상 진단, 상태 감시, FFT 분석
"""
        basis = """
RMS 값이 높다는 것은 전체 진동 에너지가 증가했다는 의미이다.
체결 불량, 베어링 이상, 배관 지지 문제, 유동 이상 등이 원인일 수 있다.
FFT 분석과 체결 상태 점검이 필요하다.
"""

    elif "FFT" in query or "주파수" in query:
        keywords = """
FFT Analysis, 주파수 분석, Frequency Analysis,
진동 진단, 이상 주파수, 고주파, 저주파, 스펙트럼 분석
"""
        basis = """
FFT 분석은 시간 영역의 진동 신호를 주파수 영역으로 변환하여 이상 원인을 구분한다.
저주파는 체결 불량이나 구조 진동, 고주파는 누설음, 베어링 이상,
Cavitation 등과 관련될 수 있다.
"""

    elif "Foreign Object" in query or "이물질" in query:
        keywords = """
Foreign Object, 이물질, 밸브 내부 이물질,
막힘, 유량 저하, 이상 소음, 압력 변화, 밸브 손상
"""
        basis = """
Foreign Object가 밸브 내부에 있으면 유로 막힘, 유량 저하, 압력 변화,
이상 소음, 밸브 시트 손상, 누설 등이 발생할 수 있다.
"""

    else:
        keywords = """
선박 배관, 밸브, 누설, 진동, 소음, 점검, 조치, 이상 진단
"""
        basis = """
선박 배관 및 밸브 이상 진단은 누설, 진동, 소음, 압력 변화,
RMS, FFT 분석 등을 종합적으로 확인하여 수행한다.
"""

    content = f"""
[Exact Query Anchor Chunk]

사용자 질문 원문:
{query}

동일 의미 질문:
- {query}
- 이 질문에 대한 원인, 증상, 점검 방법, 권장 조치는 무엇인가요?
- 관련된 선박 배관 및 밸브 이상 진단 방법은 무엇인가요?

검색 키워드:
{keywords}

답변 근거:
{basis}
"""

    return content


exact_query_docs = []

for idx, query in enumerate(query_list, start=1):
    exact_query_docs.append(
        Document(
            page_content=make_anchor_content(query),
            metadata={
                "source": "exact_query_anchor",
                "anchor_id": f"ANCHOR-{idx}",
                "original_query": query,
                "chunk_type": "exact_query_anchor_chunk"
            }
        )
    )

print(f"Exact Query Anchor Chunk 수: {len(exact_query_docs)}")

if len(exact_query_docs) > 0:
    print("\n[Exact Query Anchor 미리보기]")
    print(exact_query_docs[0].page_content[:1000])


Exact Query Anchor Chunk 수: 10

[Exact Query Anchor 미리보기]

[Exact Query Anchor Chunk]

사용자 질문 원문:
밸브에서 가스가 새요?

동일 의미 질문:
- 밸브에서 가스가 새요?
- 이 질문에 대한 원인, 증상, 점검 방법, 권장 조치는 무엇인가요?
- 관련된 선박 배관 및 밸브 이상 진단 방법은 무엇인가요?

검색 키워드:

가스 누설, 배관 누설, 밸브 누설, Gas Leakage, Pipe Leakage,
Hydrogen Gas Leak, Gas Detector, 압력 저하, 누설음, 환기, 차단, 비상정지


답변 근거:

가스 또는 배관 누설이 발생하면 가스 감지기 알람, 압력 저하, 누설음,
배관 주변 가스 감지 등의 증상이 나타날 수 있다. 권장 조치는 안전 확보,
누설 위치 표시, 밸브 격리, 환기, 비상정지, 정비 후 재점검이다.




# 22단계. Final 고정밀 Chroma VectorDB 생성

In [22]:
final_high_precision_docs = enriched_docs + faq_docs + exact_query_docs

print(f"Alias 보강 Chunk 수: {len(enriched_docs)}")
print(f"FAQ 검색 Chunk 수: {len(faq_docs)}")
print(f"Exact Query Anchor Chunk 수: {len(exact_query_docs)}")
print(f"Final 전체 Chunk 수: {len(final_high_precision_docs)}")

database_final = Chroma.from_documents(
    documents=final_high_precision_docs,
    embedding=embedding,
    collection_name=FINAL_COLLECTION_NAME,
    persist_directory=FINAL_PERSIST_DIRECTORY,
    collection_metadata={"hnsw:space": DISTANCE_METRIC}
)

print("Final Chroma DB 생성 완료")
print(f"Collection Name: {FINAL_COLLECTION_NAME}")
print(f"Persist Directory: {FINAL_PERSIST_DIRECTORY}")
print(f"Distance Metric: {DISTANCE_METRIC}")


Alias 보강 Chunk 수: 25
FAQ 검색 Chunk 수: 8
Exact Query Anchor Chunk 수: 10
Final 전체 Chunk 수: 43
Final Chroma DB 생성 완료
Collection Name: chroma-ShipLeak-final-20260616_224139
Persist Directory: ./chroma_shipleak_final_20260616_224139
Distance Metric: cosine


# 23단계. Final 성능 비교

In [23]:
final_compare = evaluate_retrieval_with_expansion(
    database=database_final,
    queries=query_list,
    k=FINAL_TOP_K,
    experiment_name="Final RAG: Alias + FAQ + Exact Query Anchor + Cosine",
    use_expansion=False
)

comparison_df_final = pd.concat(
    [
        baseline_compare,
        improved_compare,
        high_precision_compare,
        high_precision_no_expansion,
        final_compare
    ],
    ignore_index=True
)

summary_df_final = comparison_df_final.groupby("Experiment").agg(
    질문수=("Original Query", "count"),
    평균_Top_Score=("Top Score", "mean"),
    평균_Average_Score=("Average Score", "mean")
).reset_index()

summary_df_final


,Experiment,질문수,평균_Top_Score,평균_Average_Score
0,Baseline RAG,10,0.365972,0.312844
1,Final RAG: Alias + FAQ + Exact Query Anchor + ...,10,0.738209,0.652773
2,High Precision RAG: Alias + FAQ + Cosine,10,0.663796,0.576126
3,High Precision RAG: Alias + FAQ + Cosine + Que...,10,0.770917,0.672720
4,Improved RAG: Alias + Cosine + Query Expansion,10,0.650939,0.577066
